# M01C02: Identity vs. Equality; Aliasing and Object Graphs

**System Information:** Apple M1 Max (macOS), Python 3.14.0

**Version Diagnostic:**

```python:disable-run
import sys
print(f"CPython {sys.version} ({sys.implementation.cache_tag})")
assert sys.implementation.name == 'cpython', "This module assumes CPython 3.12+; adjust for alternatives."
```

> **Guidance for Extension Developers:** This module builds on CPython internals from M01C01. For portable code, focus on language-level semantics (e.g., `is` vs. `==`); avoid implementation-specific graph traversal. Consult the [Python Language Reference](https://docs.python.org/3/reference/datamodel.html) for stable behaviors.

This module elucidates the distinction between object identity (pointer equality via `is` and `id()`) and value equality (via `==` and `__eq__`), framing Python programs as directed graphs of references. Building on M01C01's header model, we explore aliasing—shared references—as a source of mutation propagation, with implications for correctness, performance, and debugging. Explanations delve into graph theory underpinnings, historical design choices, and edge cases, ensuring self-contained depth.

> **Fact Typing Convention:** Throughout this module, facts are classified as follows for clarity and longevity:
> - **Semantic Invariant:** Portable across Python implementations (e.g., PyPy); enshrined in the language specification.
> - **CPython Implementation Detail:** Specific to CPython 3.12–3.14; measured empirically; subject to evolution without notice.
> - **Forward-Looking Note:** Based on PEPs (e.g., 703); provisional until stabilized.

Structure: Integrated motivational lab, conceptual foundations, empirical explorations, common pitfalls, verification, and synthesis. All code executes in CPython 3.12+.

## Glossary of Key Terms

Essential concepts for reference; expand the section below for detailed explanations.

<details>
<summary>Expand Glossary</summary>

| Term/Abbreviation          | Definition |
|----------------------------|------------|
| **Identity** (`is`, `id()`) | Pointer equality: Determines whether two names reference the identical memory address (same PyObject header). Operates in O(1) time but yields transient results across executions due to Address Space Layout Randomization (ASLR). |
| **Equality** (`==`, `__eq__`) | Value comparison: Invokes the `__eq__` method, which performs shallow (e.g., list element matching), deep (e.g., recursive traversal), or customized evaluations. Typically O(n) in complexity. |
| **Aliasing**                | Multiple names bound to the same object (shared graph vertex); any mutation propagates across all references via directed edges. |
| **Object Graph**            | Directed graph G = (V, E): Vertices V represent objects (each with a header from M01C01), and edges E denote references from containers or namespaces. Fully encapsulates the live heap state. |
| **Reference Edge**          | Directed connection from a source (e.g., dictionary key) to a target object; the reference count approximates the in-degree plus short-lived temporaries for mortal objects. |
| **Shallow Copy**            | Duplicates the container while retaining references to existing elements (preserves subgraph sharing and edges). |
| **Deep Copy**               | Recursively duplicates the entire reachable subgraph (isolates mutations by creating new vertices). |
| **Closure Cell**            | Mutable storage slot that captures variables in nested functions or comprehensions, establishing hidden reference edges. |
| **Ownership Contract**      | Formal, documented invariant delineating permissible sharing and mutation behaviors (e.g., restricting changes to immutable subgraphs). |

</details>

## Prelude: Unmasking Aliasing Through Experiment and Scope

Python's reference model—elegant for composition—harbors pitfalls: aliasing turns local mutations global, violating intuitive isolation. This integrated prelude combines empirical demonstration with scoping, revealing the object graph as a predictive framework.

**Version Diagnostic:** Confirm environment alignment.

Focus: **CPython 3.12+** 64-bit release builds. Graph semantics align with the language spec, but details (e.g., refcount approximation of in-degree) are CPython-specific.

**Caveats:**
- Portable: `is`/`==` behaviors are standard; graphs abstract implementation.
- Extensions: Use `PyObject*` for traversal; avoid assuming refcount = exact in-degree (temporaries bias it).

**Version Notes:** No major shifts post-3.12; free-threaded builds (PEP 703) preserve graph model, enhancing concurrent safety. *(Forward-Looking Note: For 3.15+ (projected 2026), monitor PEP 744 for `__eq__` refinements in structural matching.)*

In [1]:
import sys

# Environment check
vi = sys.version_info
print(f"CPython {vi.major}.{vi.minor}")
assert sys.implementation.name == 'cpython', "CPython-specific; adjust for alternatives."

# Motivational Experiment: Mutation Through Aliases
a = [1, 2, 3]
b = a  # Alias: edge from b to a's vertex
c = [1, 2, 3]  # New vertex

print(f"id(a): {id(a):>12} | id(b): {id(b):>12} | id(c): {id(c):>12}")
print(f"a is b: {a is b:>5} | a is c: {a is c:>5} | a == c: {a == c:>5}")

b.append(999)  # Mutation traverses edge
print(f"Post-mutation: a = {a} | a == c: {a == c:>5}")

# Assertions for feedback
assert a is b, "Aliasing preserved"
assert a is not c, "Distinct vertices"
assert a != c, "Mutation alters value"
print("\nExperiment verified: Aliasing propagates; graphs predict.")

# Guiding Questions:
# - Why does b.append alter a?
# - When does = create edges vs. new vertices?
# - How to visualize and contract for safe sharing?

CPython 3.14
id(a):   4500431296 | id(b):   4500431296 | id(c):   4500433664
a is b:     1 | a is c:     0 | a == c:     1
Post-mutation: a = [1, 2, 3, 999] | a == c:     0

Experiment verified: Aliasing propagates; graphs predict.


### Identity (`is`, `id()`) vs. Equality (`==`): Foundational Distinction

**Semantic Invariant:** Identity probes the graph's vertices: `a is b` iff `id(a) == id(b)` (same memory address). Equality traverses payloads: `a == b` delegates to `a.__eq__(b)`, defaulting to content comparison for built-ins.

**CPython Implementation Detail:** `is` is pointer equality (`PyObject*` comparison); `==` calls slots via `ob_type` (e.g., `tp_as_sequence->sq_item`).

**Mnemonic Rule:** `is` for "same thing"; `==` for "same value". Limit `is` to singletons (`None`, etc.) or explicit structure.

**Interning Trap (CPython Implementation Detail):** CPython interns small ints/strings, yielding occasional `is` for literals—treat as coincidence, not rule.

**SENTINEL Idiom (Semantic Invariant):** Legitimate `is` use: `SENTINEL = object(); if val is SENTINEL:`.

In [2]:
x = [1, 2, 3]
y = [1, 2, 3]

print(f"Pre-alias: x is y = {x is y} | x == y = {x == y}")
print(f"ids: x={id(x):>12}, y={id(y):>12}")

y = x  # Alias
print(f"Post-alias: x is y = {x is y} | x == y = {x == y}")

x.append(4)  # Mutate
print(f"Post-mutation: x == [1,2,3] = {x == [1,2,3]}")

# Interning example
small_int_a = 256
small_int_b = 256
print(f"Small ints: {small_int_a is small_int_b} (interned; unreliable)")

short_str_a = "hello"
short_str_b = "hello"
print(f"Short strings: {short_str_a is short_str_b} (interned; unreliable)")

# SENTINEL
SENTINEL = object()
def process(val=SENTINEL):
    if val is SENTINEL:
        return "default"
process()  # Works

# Insight: is invariant; == dynamic; intern traps mislead.

Pre-alias: x is y = False | x == y = True
ids: x=  4500565888, y=  4500555264
Post-alias: x is y = True | x == y = True
Post-mutation: x == [1,2,3] = False
Small ints: True (interned; unreliable)
Short strings: True (interned; unreliable)


'default'

## The Object Graph: Python State as Directed Graph

**Semantic Invariant:** The heap is modeled as G = (V, E): V = objects; E = references (directed edges). This abstraction predicts mutation propagation regardless of implementation.

**CPython Implementation Detail:** Refcount ≈ |{incoming edges}| + temporaries (for mortals; noisy probe). Immortal singletons (e.g., empty tuple `()`, `frozenset()`) are shared globally with sentinel refcounts, eliding increments—only one vertex per type.

**Primary Edge Sources:**

| Category | Sources | Examples | Persistence |
|----------|---------|----------|-------------|
| Namespaces | Bindings | locals/globals, imports | Scope lifetime |
| Containers | Slots | list[i], dict[key] | Until mutation/del |
| Fields | Attrs | __dict__, __slots__ | Instance lifetime |
| Callables | Baggage | __defaults__, __closure__ | Function lifetime |

**Notes:** Omits weakrefs (non-refcounted), C-extension holds, frame links—focus on Python-visible strong edges. *(Preview: Weakrefs and GC in M01C05.)*

**Rationale:** Graphs model sharing; refcount aids debugging but isn't exact law.

**Lab 1: Tracing Edge Creation and Refcount Impact**

Add edges; observe deltas (+1 per long-lived; temps vary). **Never write production logic that depends on exact refcount values.**

**Demonstration of Temporaries Bias:** `sys.getrefcount` includes a call-frame temporary (+1), so initial count is 2 for a lone variable.

In [3]:
import sys
def refcount(o): return sys.getrefcount(o) - 1  # Approx; temps noisy

# Concrete temporary example
x = []
print(f"x = []: sys.getrefcount(x) = {sys.getrefcount(x)}  # 1 from 'x', 1 from getrefcount temporary")
def f(l=x): pass
print(f"After def f(l=x): pass: sys.getrefcount(x) = {sys.getrefcount(x)}  # +1 from f.__defaults__")

head = ['root']
print(f"Baseline (head): {refcount(head)}")

tail = head  # Namespace edge
print(f"After alias (tail): {refcount(head)}")

node = {'next': head}  # Container edge
print(f"After dict slot: {refcount(head)}")

def factory(x=head): pass  # Default arg edge (permanent)
print(f"After default arg: {refcount(head)}")

head.append(head)  # Self-ref edge
print(f"After self-append: {refcount(head)}")

# Expected: +1 per edge; temps vary—graph model trumps probe.

x = []: sys.getrefcount(x) = 2  # 1 from 'x', 1 from getrefcount temporary
After def f(l=x): pass: sys.getrefcount(x) = 3  # +1 from f.__defaults__
Baseline (head): 1
After alias (tail): 2
After dict slot: 3
After default arg: 4
After self-append: 5


In [4]:
# Immortal singleton demo (CPython Implementation Detail; PEP 683)
empty_tuple = ()
another_tuple = ()
print(f"Empty tuple shared: {empty_tuple is another_tuple}")

empty_frozenset = frozenset()
another_frozenset = frozenset()
print(f"Empty frozenset shared: {empty_frozenset is another_frozenset}")

if hasattr(sys, '_is_immortal'):
    print(f"Empty tuple immortal: {sys._is_immortal(empty_tuple)}")

# Note: Single global vertex; refcount sentinel—optimizes long-running interpreters.

Empty tuple shared: True
Empty frozenset shared: False
Empty tuple immortal: True


### Visualization: Simple Object Graph

<!-- Placeholder for diagram; in practice, use matplotlib or ASCII -->

```
  +-------+     +-------+
  |  'a'  | --> | List: |
  +-------+     | [1,2] |
                +-------+
                   ^
                   |
  +-------+     +-------+
  |  'b'  | --> |  1   |
  +-------+     +-------+
                   ^
                   |
  +-------+     +-------+
  |  'c'  | --> |  2   |
  +-------+     +-------+
```

**Key:** Namespaces (a,b,c) → vertices; list slots → edges. Mutation at vertex affects all incoming.

## Common Pitfalls: Hidden Edges and Aliasing Traps

**Trap 1: Mutable Defaults**

`__defaults__` binds mutable defaults permanently—one edge per function instance.

**Failure:**
```python
def bad_tree(init_nodes=[]):
    init_nodes.append('new')
    return init_nodes
```
**Fix:** Sentinel check; init inside.

In [5]:
def bad_tree(init_nodes=[]):  # Shared default!
    init_nodes.append('new')
    return init_nodes

t1 = bad_tree()
t2 = bad_tree()

print(f"Bad: t1 = {t1}, t2 = {t2}, shared: {t1 is t2}")

# Fixed
def good_tree(init_nodes=None):
    if init_nodes is None:
        init_nodes = []
    init_nodes.append('new')
    return init_nodes

g1 = good_tree()
g2 = good_tree()
print(f"Good: g1 = {g1}, g2 = {g2}, separate: {g1 is not g2}")

Bad: t1 = ['new', 'new'], t2 = ['new', 'new'], shared: True
Good: g1 = ['new'], g2 = ['new'], separate: True


**Trap 2: Late Binding Closures**

Comprehensions capture final loop var—one edge to the end-state.

**Failure:**
```python
funcs = [lambda x: x + i for i in range(3)]  # All bind final i=2
```
**Fix:** Default arg shadows.

In [6]:
funcs = [lambda x: x + i for i in range(3)]  # All bind final i=2
print([f(10) for f in funcs])  # [12,12,12]

# Fixed
funcs_fixed = [lambda x, i=i: x + i for i in range(3)]
print([f(10) for f in funcs_fixed])  # [10,11,12]

[12, 12, 12]
[10, 11, 12]


**Modern Pitfall Sidebar:** In `dataclasses` (PEP 557), `field(default_factory=[]) ` avoids mutable defaults; `frozen=True` enforces immutability for sharing. For `asyncio` tasks, `asyncio.create_task` shares the coroutine object—use `copy` for isolation. See [PEP 572](https://peps.python.org/pep-0572/) for assignment expressions' local binding edges.

## Verification: Core Graph Invariants

**Semantic Invariant:** Ownership contracts specify allowed sharing (e.g., immutable subgraphs) and forbidden aliasing (e.g., no cross-list edges in disjoint DS).

Asserts ownership contracts; detects aliasing violations. For a linked list DS: "'next' edges form a DAG; no sharing across independent lists—violation corrupts lengths."

**Coverage:** Sharing corruption, intentional edges, copy isolation.

In [7]:
import copy

class Node:
    def __init__(self, value, next=None):
        self.value = value
        self.next = next

def length(head):
    count = 0
    cur = head
    while cur:
        count += 1
        cur = cur.next
    return count

def test_aliasing_corruption():
    """Forbidden: Cross-list sharing corrupts independent lengths."""
    shared = Node(10, Node(20))
    list1 = Node(1, Node(2, shared))
    list2 = Node(5, shared)
    # Pre-contract: Disjoint lengths
    assert length(list1) == 4
    assert length(list2) == 3
    list1.next.next.next = None  # Violate: Mutate shared
    assert length(list1) == 3
    assert length(list2) == 2  # Breach: list2 shortened

def test_intentional_sharing():
    """Allowed: Shared immutable tail (contract: no mutation post-share)."""
    shared = Node('shared')
    a = Node('A', shared)
    b = Node('B', shared)
    assert a.next is b.next  # Contract: shared immutable

def test_deepcopy_isolates():
    """Deep copy: New vertices; isolates mutations."""
    orig = Node(1, Node(2))
    dup = copy.deepcopy(orig)
    orig.next.value = 999
    assert dup.next.value == 2  # Isolated (memo preserves aliases if intended)

def test_shallow_preserves_sharing():
    """Shallow copy: Preserves edges to subgraphs."""
    inner = Node('inner')
    orig = Node('orig', inner)
    shallow = copy.copy(orig)
    inner.value = 'mutated'
    assert shallow.next.value == 'mutated'

# Run core
suite = [test_aliasing_corruption, test_intentional_sharing, test_deepcopy_isolates, test_shallow_preserves_sharing]
for test in suite:
    test()
print("Core invariants hold.")

# Note: For adversarial testing, use Hypothesis to generate random graphs violating contracts (requires `pip install hypothesis`).

Core invariants hold.


**Tiered Exercise: Binary Tree Sharing and Cloning**

**Tier 1 (Basic):** Implement Node; build sharing trees; verify `is` sharing.

**Tier 2 (Intermediate):** Mutate shared; confirm visibility; add cycle check (DFS). Spec: "Left/right subtrees may share immutable leaves; no cycles."

**Tier 3 (Advanced):** Manual `clone()` (recursive); profile vs. `deepcopy` on 10K-node DAG. Property: Post-clone, mutations isolate; sharing preserved only for memoized.

**Hint:** Use `dataclasses` with `frozen=True` for immutable nodes; `networkx` for DFS.

**Spot-the-Bug: Concat Trap**

Destructive splice violates nondestructive contract: "concat returns new list; original unchanged."

**Failing Test:**
```python
def concat(a, b):
    if not a: return b
    cur = a
    while cur.next: cur = cur.next
    cur.next = b
    return a

orig = Node(1, Node(2))
result = concat(orig, Node(3))
assert orig.next.next.value == 3  # Breach: orig mutated!
```
**Fix:** Immutable: `return Node(a.value, concat(a.next, b))`.

## Essential Takeaways: Graph Invariants

| Invariant | Implication | Mitigation |
|-----------|-------------|------------|
| Heap as directed graph | Mutations propagate via edges | Trace with `gc.get_referents()` |
| `is` vs. `==` | Vertex vs. value checks | Use `is` for singletons only |
| Aliasing = shared vertices | Global visibility | Immutable subgraphs or contracts |
| Edge sources primary | Hidden aliases from defaults/closures | Explicit init; pass args |
| Shallow/deep copy | Preserve vs. isolate edges | Deep for isolation; shallow for perf |
| Refcount ≈ in-degree | + temps; weakrefs exempt | Use `sys.getrefcount` cautiously |

**Synthesis:** Graphs predict sharing; contracts enforce invariants—foundation for mutability (C03).

## Progression: Next in M01

- **M01C03: Mutability vs. Immutability** – Edge addition/removal; views/copies.
- **M01C05: Cycle GC** – Graph cycles beyond refcount.

Graph mastery unmasks mutation surprises.

## Appendix: Advanced Tests and Full Solutions

**Preview: Weakrefs and Concurrent** – Non-strong edges; GIL propagation. (Full story in M01C05.)

**Binary Tree Solution:** Mutable Node for demo; clone isolates.

In [8]:
import copy, sys, weakref, threading, networkx as nx
from dataclasses import dataclass
from typing import Optional

# Advanced Tests (Preview)
class Node:
    def __init__(self, value, next=None):
        self.value = value
        self.next = next

def test_weakref_noncounting_edge():
    obj = Node(42)
    wr = weakref.ref(obj)
    assert wr() is obj
    del obj
    assert wr() is None

def test_concurrent_aliasing():
    shared = []
    def mutate1(): shared.append('from1')
    def mutate2(): shared.append('from2')
    t1 = threading.Thread(target=mutate1)
    t2 = threading.Thread(target=mutate2)
    t1.start(); t2.start(); t1.join(); t2.join()
    assert len(shared) == 2

test_weakref_noncounting_edge()
test_concurrent_aliasing()
print("Advanced: Weak/concurrent hold.")

# Binary Tree Solution (Tiers 1–3)
@dataclass  # Mutable for demo
class TreeNode:
    val: int
    left: Optional['TreeNode'] = None
    right: Optional['TreeNode'] = None

    def clone(self) -> 'TreeNode':
        return TreeNode(self.val, self.left.clone() if self.left else None, self.right.clone() if self.right else None)

    def has_cycle(self) -> bool:
        G = nx.DiGraph()
        def build(g, node):
            nid = id(node)
            if nid in g: return
            g.add_node(nid)
            if node.left:
                g.add_edge(nid, id(node.left))
                build(g, node.left)
            if node.right:
                g.add_edge(nid, id(node.right))
                build(g, node.right)
        build(G, self)
        return not nx.is_directed_acyclic_graph(G)

def build_shared_trees():
    shared_4 = TreeNode(4, TreeNode(5), TreeNode(6))
    tree1 = TreeNode(1, TreeNode(2), shared_4)
    tree2 = TreeNode(10, TreeNode(20), shared_4)
    return tree1, tree2, shared_4

tree1, tree2, shared = build_shared_trees()
assert tree1.right is tree2.right  # Tier 1
tree1.right.left.val = 999  # Tier 2
assert tree2.right.left.val == 999
assert not tree1.has_cycle()
clone = tree1.clone()  # Tier 3
tree1.right.left.val = 111
assert clone.right.left.val == 999
print("Tree Solution: Tiers complete.")

# Concat Fix
def concat_immutable(a, b):
    if not a: return b
    return Node(a.value, concat_immutable(a.next, b))

orig = Node(1, Node(2))
result = concat_immutable(orig, Node(3))
assert orig.next.value == 2  # Unchanged
print("Concat fixed: Immutable.")

Advanced: Weak/concurrent hold.
Tree Solution: Tiers complete.
Concat fixed: Immutable.
